#Problem Statement:

Postpartum depression (PPD) is a serious but often underdiagnosed condition in new mothers, negatively impacting both mother and child. Early detection is vital for effective intervention. This project aims to build a machine learning classification model using a dataset on symptoms and behaviors to identify mothers at "High Risk" or "Low Risk" for PPD. The goal is to create an accurate tool to help healthcare providers with early screening and support.


























































































##Importing Dataset

In [ ]:
pip install ydata_profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.4/400.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 171.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 153.4 MB/s eta 0:00:00
  Attempting uninstall: llvmlite
    Found e

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ydata_profiling as yp
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_squared_error

Loading Dataset

In [ ]:
dataset = pd.read_csv("/content/postpartum dataset.csv")

##Exploratory Data analysis

In [ ]:
profile =yp.ProfileReport(dataset, title="EDA Report", explorative =True)
profile.to_file("eda_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 11/11 [00:00<00:00, 85281.60it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

##Data Preprocessing


In [ ]:
if "Timestamp" in dataset.columns:
    dataset = dataset.drop(columns=["Timestamp"])

In [ ]:
dataset.head()

,Age,Feeling sad or Tearful,Irritable towards baby & partner,Trouble sleeping at night,Problems concentrating or making decision,Overeating or loss of appetite,Feeling anxious,Feeling of guilt,Problems of bonding with baby,Suicide attempt
0,35-40,Yes,Yes,Two or more days a week,Yes,Yes,Yes,No,Yes,Yes
1,40-45,Yes,No,No,Yes,Yes,No,Yes,Yes,No
2,35-40,Yes,No,Yes,Yes,Yes,Yes,No,Sometimes,No
3,35-40,Yes,Yes,Yes,Yes,No,Yes,Maybe,No,No
4,40-45,Yes,No,Two or more days a week,Yes,No,Yes,No,Yes,No


In [ ]:
print(dataset.isna().sum())

Age                                           0
Feeling sad or Tearful                        0
Irritable towards baby & partner              6
Trouble sleeping at night                     0
Problems concentrating or making decision    12
Overeating or loss of appetite                0
Feeling anxious                               0
Feeling of guilt                              9
Problems of bonding with baby                 0
Suicide attempt                               0
dtype: int64


In [ ]:
duplicates_count = dataset.duplicated().sum()
print(f"Number of duplicate rows: {duplicates_count}")
if duplicates_count > 0:
    dataset = dataset.drop_duplicates()
    print(f"Duplicates removed. New shape: {dataset.shape}")
else:
    print("No duplicate rows found.")

Number of duplicate rows: 1177
Duplicates removed. New shape: (326, 10)


Convert categorical symptom answers (Yes/No) to numeric

In [ ]:
dataset_encoded = dataset.copy()
for col in dataset.columns:
    if dataset_encoded[col].dtype == "object":
        dataset_encoded[col] = dataset_encoded[col].astype(str).str.strip().str.lower()
        mapping = {"yes": 2, "no": 0, "Two or more days a week": 2.5, "sometimes": 1.5, "maybe": 1, "always": 3}
        dataset_encoded[col] = dataset_encoded[col].map(mapping).fillna(dataset_encoded[col])
        dataset_encoded[col] = pd.to_numeric(dataset_encoded[col], errors="coerce").fillna(0)

Create depression score (sum of all symptom items except Age)

In [ ]:
symptom_cols = [c for c in dataset_encoded.columns if c.lower() not in ["age", "depression_label", "depression_score"]]
print("Columns used for depression score calculation:", symptom_cols)
dataset_encoded["depression_score"] = dataset_encoded[symptom_cols].sum(axis=1)

Columns used for depression score calculation: ['Feeling sad or Tearful', 'Irritable towards baby & partner', 'Trouble sleeping at night', 'Problems concentrating or making decision', 'Overeating or loss of appetite', 'Feeling anxious', 'Feeling of guilt', 'Problems of bonding with baby', 'Suicide attempt']


Choose a threshold (example: score >= 7 means "high risk")

In [ ]:
threshold = 7
dataset_encoded["depression_label"] = (dataset_encoded["depression_score"] >= threshold).astype(int)

Checking label balance

In [ ]:
print("\nLabel distribution:")
print(dataset_encoded["depression_label"].value_counts())


Label distribution:
depression_label
1    248
0     78
Name: count, dtype: int64


## Split data


In [ ]:
X = dataset_encoded.drop(columns=["depression_label", "depression_score"])
y = dataset_encoded["depression_label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Data split into training and testing sets.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Data split into training and testing sets.
X_train shape: (260, 10)
X_test shape: (66, 10)
y_train shape: (260,)
y_test shape: (66,)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(),
    "SVC": SVC()
}

In [ ]:
for name, model in models.items():
    print(f"Training {name} with engineered features...")
    model.fit(X_train, y_train) # Use X_train and y_train with engineered features
    models[name] = model
    print(f"{name} trained.")

Training Logistic Regression with engineered features...
Logistic Regression trained.
Training Random Forest with engineered features...
Random Forest trained.
Training SVC with engineered features...
SVC trained.


In [ ]:
from sklearn.metrics import mean_squared_error

results_engineered = {} # Store results with engineered features separately
for name, model in models.items():
    y_pred = model.predict(X_test) # Use X_test with engineered features
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    results_engineered[name] = {"accuracy": accuracy, "precision": precision, "recall": recall, "f1_score": f1, "mse": mse}
    print(f"\n{name} Performance (Engineered Features):")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-score: {f1:.4f}")
    print(f"  MSE: {mse:.4f}")


Logistic Regression Performance (Engineered Features):
  Accuracy: 1.0000
  Precision: 1.0000
  Recall: 1.0000
  F1-score: 1.0000
  MSE: 0.0000

Random Forest Performance (Engineered Features):
  Accuracy: 0.9091
  Precision: 0.9074
  Recall: 0.9800
  F1-score: 0.9423
  MSE: 0.0909

SVC Performance (Engineered Features):
  Accuracy: 0.9545
  Precision: 0.9608
  Recall: 0.9800
  F1-score: 0.9703
  MSE: 0.0455


In [ ]:
results_df = pd.DataFrame.from_dict(results_engineered, orient="index")
print("\nComparison Table of Model Performance (Engineered Features):")
display(results_df)


Comparison Table of Model Performance (Engineered Features):


,accuracy,precision,recall,f1_score,mse
Logistic Regression,1.000000,1.000000,1.00,1.000000,0.000000
Random Forest,0.909091,0.907407,0.98,0.942308,0.090909
SVC,0.954545,0.960784,0.98,0.970297,0.045455
